# Target Monitoring

This notebook produces scores to monitor the accuracy of performance targets generated by marketing analytics models.

In [ ]:
from datetime import datetime

import numpy as np
import pandas as pd

from modules import query_tools

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

## Glossary

- **Combination**: a grouping of channel, product, country group, device class, and acquisition type.
- **Metric change**: the absolute difference between short-term and longer-term target performance.

## Component 1: Cost Score

This component scores combinations based on their share of total spend.

Steps:

- Pull weekly cost by combination
- Aggregate cost across the reporting window
- Calculate each combination's share of total cost
- Convert cost share into a score from 0 to 3

In [ ]:
COST_COLS_DTYPES = {
    "total_cost_usd": "float64"
}

weekly_combination_costs = (
    query_tools
    .run_query("get/get_weekly_combination_costs.sql")
    .astype(COST_COLS_DTYPES)
)

In [ ]:
combination_costs = (
    weekly_combination_costs
    .groupby([
        "channel_name",
        "product_name",
        "device_class",
        "acquisition_type",
        "country_group"
    ], as_index=False)
    .agg({"total_cost_usd": "sum"})
)

combination_costs["pct_total_cost_usd"] = (
    combination_costs["total_cost_usd"] /
    combination_costs["total_cost_usd"].sum()
)

combination_costs["cost_score"] = (
    3 * combination_costs["pct_total_cost_usd"] /
    combination_costs["pct_total_cost_usd"].max()
)

## Component 2: Absolute Metric Change Score

This component measures the average absolute difference between short-term and longer-term ROI-vs-target performance.

Score logic:

- Metric change >= 20% → score of 2
- 10% <= metric change < 20% → score of 1
- 0% <= metric change < 10% → score of 0.5
- Otherwise → score of 0

In [ ]:
ROI_TARGET_COLS_DTYPES = {
    "total_cost_usd": "float64",
    "avg_roi_vs_target_4d": "float64",
    "avg_roi_vs_target_90d": "float64"
}

weekly_roi_vs_target = (
    query_tools
    .run_query("get/get_weekly_roi_vs_target.sql")
    .fillna(0)
    .astype(ROI_TARGET_COLS_DTYPES)
)

In [ ]:
def calculate_absolute_metric_change(short_term_value, long_term_value):
    if long_term_value:
        metric_change = abs((short_term_value / long_term_value) - 1)
    else:
        metric_change = 0

    return metric_change

In [ ]:
weekly_absolute_metric_change = weekly_roi_vs_target.copy(deep=True)

weekly_absolute_metric_change["abs_metric_change"] = (
    weekly_absolute_metric_change.apply(
        lambda row: calculate_absolute_metric_change(
            row["avg_roi_vs_target_4d"],
            row["avg_roi_vs_target_90d"]
        ),
        axis=1
    )
)

In [ ]:
absolute_metric_change = (
    weekly_absolute_metric_change
    .groupby([
        "channel_name",
        "product_name",
        "device_class",
        "acquisition_type",
        "country_group"
    ], as_index=False)
    .agg({"abs_metric_change": "mean"})
    .rename(columns={
        "abs_metric_change": "avg_abs_metric_change"
    })
)

absolute_metric_change["metric_absolute_score"] = np.where(
    absolute_metric_change["avg_abs_metric_change"] >= 0.2,
    2,
    np.where(
        absolute_metric_change["avg_abs_metric_change"] >= 0.1,
        1,
        np.where(
            absolute_metric_change["avg_abs_metric_change"] >= 0,
            0.5,
            0
        )
    )
)

## Component 3: Volatility Score

This component scores combinations based on how volatile the metric change is over time.

Steps:

- Calculate standard deviation of weekly absolute metric change
- Count reporting periods
- Scale volatility by the square root of the number of periods
- Convert volatility into a score from 0 to 3

In [ ]:
metric_change_stddev = (
    weekly_absolute_metric_change
    .copy(deep=True)
    .loc[:, [
        "channel_name",
        "product_name",
        "device_class",
        "acquisition_type",
        "country_group",
        "abs_metric_change"
    ]]
    .groupby([
        "channel_name",
        "product_name",
        "device_class",
        "acquisition_type",
        "country_group"
    ], as_index=False)
    .agg(np.std, ddof=0)
    .rename(columns={
        "abs_metric_change": "stddev_abs_metric_change"
    })
)

metric_change_periods = (
    weekly_absolute_metric_change
    .groupby([
        "channel_name",
        "product_name",
        "device_class",
        "acquisition_type",
        "country_group"
    ], as_index=False)
    .agg({"abs_metric_change": "count"})
    .rename(columns={
        "abs_metric_change": "count_time_periods"
    })
)

metric_change_periods["sqrt_count_time_periods"] = (
    np.sqrt(metric_change_periods["count_time_periods"])
)

metric_volatility = metric_change_stddev.merge(
    metric_change_periods,
    how="left",
    on=[
        "channel_name",
        "product_name",
        "device_class",
        "acquisition_type",
        "country_group"
    ]
)

metric_volatility["metric_volatility"] = (
    metric_volatility["stddev_abs_metric_change"]
    * metric_volatility["sqrt_count_time_periods"]
)

metric_volatility["volatility_score"] = (
    3 * metric_volatility["metric_volatility"] /
    metric_volatility["metric_volatility"].max()
)

## Component 4: Duration Score

This component scores combinations based on how often they exceed a metric-change threshold.

Steps:

- Flag weeks where absolute metric change is above 20%
- Count flagged weeks by combination
- Assign a score of 2 when at least 4 weeks exceed the threshold
- Assign 0 otherwise

In [ ]:
metric_change_duration = weekly_absolute_metric_change.copy(deep=True)

metric_change_duration["has_metric_change_more_than_20pct"] = np.where(
    metric_change_duration["abs_metric_change"] > 0.2,
    1,
    0
)

metric_change_duration_count = (
    metric_change_duration
    .groupby([
        "channel_name",
        "product_name",
        "device_class",
        "acquisition_type",
        "country_group"
    ], as_index=False)
    .agg({"has_metric_change_more_than_20pct": "sum"})
    .rename(columns={
        "has_metric_change_more_than_20pct": "periods_above_threshold"
    })
)

metric_change_duration_count["duration_score"] = np.where(
    metric_change_duration_count["periods_above_threshold"] >= 4,
    2,
    0
)

## Review Priority Score

The final score combines cost, absolute metric change, volatility, and duration scores.

The score is designed to help prioritise which channel/product/country/device combinations should be reviewed first.

In [ ]:
combination_scores = (
    combination_costs
    .merge(
        absolute_metric_change,
        how="left",
        on=[
            "product_name",
            "channel_name",
            "device_class",
            "acquisition_type",
            "country_group"
        ]
    )
    .merge(
        metric_volatility,
        how="left",
        on=[
            "product_name",
            "channel_name",
            "device_class",
            "acquisition_type",
            "country_group"
        ]
    )
    .merge(
        metric_change_duration_count,
        how="left",
        on=[
            "product_name",
            "channel_name",
            "device_class",
            "acquisition_type",
            "country_group"
        ]
    )
)

combination_scores["review_priority_score"] = (
    combination_scores["cost_score"]
    + combination_scores["metric_absolute_score"]
    + combination_scores["volatility_score"]
    + combination_scores["duration_score"]
)

combination_scores["calc_date"] = pd.to_datetime(
    datetime.today().strftime("%Y-%m-%d")
)

In [ ]:
reporting_window_dates = (
    weekly_combination_costs
    .loc[:, [
        "reporting_window_start_date",
        "reporting_window_end_date"
    ]]
    .drop_duplicates()
    .values[0]
)

combination_scores[[
    "reporting_window_start_date",
    "reporting_window_end_date"
]] = reporting_window_dates

# Upload to BigQuery

In [ ]:
SCORES_TABLE_COLUMNS = {
    "product_name": "object",
    "channel_name": "object",
    "device_class": "object",
    "acquisition_type": "object",
    "country_group": "object",
    "total_cost_usd": "float64",
    "pct_total_cost_usd": "float64",
    "cost_score": "float64",
    "avg_abs_metric_change": "float64",
    "metric_absolute_score": "float64",
    "stddev_abs_metric_change": "float64",
    "count_time_periods": "int64",
    "metric_volatility": "float64",
    "volatility_score": "float64",
    "periods_above_threshold": "int64",
    "duration_score": "float64",
    "review_priority_score": "float64",
    "calc_date": "datetime64[ns]",
    "reporting_window_start_date": "datetime64[ns]",
    "reporting_window_end_date": "datetime64[ns]"
}

combination_scores_formatted = (
    combination_scores
    .astype(SCORES_TABLE_COLUMNS)
    .loc[:, SCORES_TABLE_COLUMNS.keys()]
)

## Check if daily run has already executed

In [ ]:
SCORES_TABLE_ID = "your-gcp-project-id.monitoring_dataset.performance_scores_v3"

daily_run_rowcount = query_tools.run_query(
    "get/check_today_scores_exist.sql"
)

daily_run_executed_already = bool(
    daily_run_rowcount["count_rows"][0]
)

if daily_run_executed_already:
    print("The daily run has already been executed. New data not uploaded.")
else:
    print(f"Uploading data to table: {SCORES_TABLE_ID}.")
    combination_scores_formatted.to_bq(SCORES_TABLE_ID)
    print("Data successfully uploaded to the table.")